In [1]:
from pyspark.sql.functions import col, to_timestamp

# Read all Bronze JSON files
bronze_df = spark.read \
    .option("multiLine", "true") \
    .json("s3a://weather-pipeline-036815016562-ap-south-1-an/bronze/*.json")

# Transform
silver_df = bronze_df.select(
    col("name").alias("city"),
    col("sys.country").alias("country"),
    col("coord.lat").alias("latitude"),
    col("coord.lon").alias("longitude"),
    col("main.temp").alias("temperature"),
    col("main.feels_like").alias("feels_like"),
    col("main.temp_min").alias("temp_min"),
    col("main.temp_max").alias("temp_max"),
    col("main.humidity").alias("humidity"),
    col("main.pressure").alias("pressure"),
    col("wind.speed").alias("wind_speed"),
    col("wind.deg").alias("wind_direction"),
    col("clouds.all").alias("cloudiness"),
    col("visibility"),
    col("weather")[0]["main"].alias("weather"),
    col("weather")[0]["description"].alias("description"),
    to_timestamp("ingestion_time").alias("ingestion_time"),
    col("source")
).dropDuplicates()

# Save as Delta Table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_weather")

display(spark.table("silver_weather"))

ModuleNotFoundError: No module named 'pyspark'

In [0]:
%sql
select * from silver_weather where city = 'Chennai';

city,country,latitude,longitude,temperature,feels_like,temp_min,temp_max,humidity,pressure,wind_speed,wind_direction,cloudiness,visibility,weather,description,ingestion_time,source
Chennai,IN,13.0878,80.2785,32.31,39.14,32.2,32.74,64,1008,0.89,230,100,10000,Clouds,overcast clouds,2026-07-10T12:37:04.000Z,OpenWeatherMap
Chennai,IN,13.0878,80.2785,35.54,42.54,34.49,36.67,52,1007,3.13,186,100,10000,Clouds,overcast clouds,2026-07-09T11:24:59.000Z,OpenWeatherMap
Chennai,IN,13.0878,80.2785,36.77,43.77,36.65,39.57,51,1003,1.34,135,70,10000,Clouds,broken clouds,2026-07-13T16:02:16.000Z,OpenWeatherMap
Chennai,IN,13.0878,80.2785,33.58,39.16,32.27,33.87,55,1006,0.89,238,91,10000,Clouds,overcast clouds,2026-07-13T09:25:57.000Z,OpenWeatherMap
